In [1]:
# RAG SGU - PyPDFLoader Toi Gian
print("Flow: config -> ingest -> chunk -> vector store -> query")

Flow: config -> ingest -> chunk -> vector store -> query


In [2]:
import sys
from pathlib import Path

from dotenv import load_dotenv

BASE_DIR = Path.cwd()
SRC_DIR = BASE_DIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from rag_core import RAGConfig, RAGPipeline, configure_runtime_environment

configure_runtime_environment()
load_dotenv(BASE_DIR / ".env")

print(f"Workspace: {BASE_DIR}")
print(f"SRC path: {SRC_DIR}")

C:\Users\quoch\AppData\Roaming\Python\Python310\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Workspace: a:\RAG-SGU
SRC path: a:\RAG-SGU\src


In [3]:
import os


os.environ["RAG_PDF_DIR"] = str(BASE_DIR / "File_PDFs_OCR")

config = RAGConfig.from_env(base_dir=BASE_DIR)
pipeline = RAGPipeline(config)

print("Configuration loaded:")
print(f"- Raw env LLM_TEMPERATURE: {os.getenv('LLM_TEMPERATURE')}")
print(f"- PDF dir: {config.pdf_dir}")
print(f"- Vector store: {config.vector_store_dir}")
print(f"- Chunk size: {config.chunk_size}")
print(f"- Chunk overlap: {config.chunk_overlap}")
print(f"- Retrieval k: {config.retrieval_k}")
print(f"- Embedding model: {config.embedding_model}")
print(f"- LLM model: {config.llm_model}")
print(f"- LLM temperature: {config.llm_temperature}")
print(f"- LLM max tokens: {config.llm_max_tokens}")

Configuration loaded:
- Raw env LLM_TEMPERATURE: None
- PDF dir: a:\RAG-SGU\File_PDFs_OCR
- Vector store: A:\RAG-SGU\vector_store
- Chunk size: 1600
- Chunk overlap: 200
- Retrieval k: 4
- Embedding model: sentence-transformers/paraphrase-multilingual-mpnet-base-v2
- LLM model: gemini-2.5-flash
- LLM temperature: 0.5
- LLM max tokens: 1024


In [4]:
build_result = None

try:
    build_result = pipeline.build_index()
    print("Index built successfully")
    print(f"- Loaded PDFs: {len(build_result.ingestion.loaded_pdf_files)}")
    print(f"- Extracted docs: {len(build_result.ingestion.documents)}")
    print(f"- Created chunks: {len(build_result.chunks)}")

    if build_result.ingestion.scanned_pdf_files:
        print("Warning: Some PDFs have no text-layer (skipped):")
        for file_name in build_result.ingestion.scanned_pdf_files:
            print(f"  - {file_name}")
except ValueError as exc:
    print("Could not build from PDFs:")
    print(exc)
    print("Trying to load existing FAISS index in next cell...")

  0%|          | 0/2 [00:00<?, ?it/s]C:\Users\quoch\AppData\Roaming\Python\Python310\site-packages\pypdf\_crypt_providers\_cryptography.py:32: CryptographyDeprecationWarning: ARC4 has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.ARC4 and will be removed from cryptography.hazmat.primitives.ciphers.algorithms in 48.0.0.
  from cryptography.hazmat.primitives.ciphers.algorithms import AES, ARC4
100%|██████████| 2/2 [00:14<00:00,  7.11s/it]


Index built successfully
- Loaded PDFs: 2
- Extracted docs: 90
- Created chunks: 162


## Build or Load Index


In [5]:
if build_result is None:
    pipeline.load_index()
    print("Loaded existing index from disk")
else:
    print("Using freshly built in-memory index")

def ask(question: str):
    result = pipeline.query(question)
    print(f"Question: {question}")
    print(f"Answer: {result['answer']}")

    if result["sources"]:
        print("Sources:")
        for source in result["sources"]:
            print(f"  - {source}")
    else:
        print("Sources: (none)")

    return result

Using freshly built in-memory index


In [6]:
# Override ask() to show detailed retrieved passages in notebook output.
def _source_detail(doc, idx: int) -> str:
    metadata = getattr(doc, "metadata", {}) or {}
    source = (
        metadata.get("source")
        or metadata.get("source_relpath")
        or metadata.get("source_path")
        or metadata.get("file_name")
    )

    page = metadata.get("page_number")
    if page is None and metadata.get("page") is not None:
        try:
            page = int(metadata.get("page")) + 1
        except (TypeError, ValueError):
            page = metadata.get("page")
    elif page is not None:
        try:
            page = int(page)
        except (TypeError, ValueError):
            pass

    label = Path(str(source)).name if source else f"Tài liệu {idx} (thiếu metadata nguồn)"
    if page is not None:
        label = f"{label} - trang {page}"

    content = str(getattr(doc, "page_content", "")).replace("\n", " ").strip()
    preview = content[:180] + ("..." if len(content) > 180 else "")
    return f"{label}: {preview}" if preview else label

def ask(question: str, show_doc_details: bool = True, max_doc_details: int = 5):
    result = pipeline.query(question)
    print(f"Question: {question}")
    print(f"Answer: {result['answer']}")

    if result["sources"]:
        print("Sources:")
        for source in result["sources"]:
            print(f"  - {source}")
    else:
        print("Sources: (none)")

    if show_doc_details and result.get("docs"):
        print("\nTop retrieved passages:")
        for idx, doc in enumerate(result["docs"][:max_doc_details], start=1):
            print(f"  {idx}. {_source_detail(doc, idx)}")

    return result

In [7]:
demo_result = ask("Muc tieu dao tao cua nganh Cong nghe thong tin la gi?")

Question: Muc tieu dao tao cua nganh Cong nghe thong tin la gi?
Answer: Dựa trên thông tin từ tài liệu, mục tiêu đào tạo của ngành Công nghệ thông tin là:

*   **7. Có cơ hội việc làm và học tập nâng cao trình độ sau tốt nghiệp ngành Công nghệ thông tin**
    *   **7.1. Cơ hội việc làm sau tốt nghiệp**
        *   Lập trình viên, kiểm thử viên, quản trị cơ sở dữ liệu, quản trị mạng, nhân viên tin học, quản trị website ở các công ty đơn vị như ngân hàng, công ty chứng khoán, công ty truyền thông, bưu điện, trường học.
        *   Tư vấn viên, cung cấp giải pháp thiết kế bảo mật, xây dựng bảo mật, dịch vụ an toàn dữ liệu ở các công ty tư vấn giải pháp kỹ thuật cao trong CNTT.
        *   Tham gia vào các công đoạn của việc phát triển phần mềm ở các công ty phần mềm.
    *   **7.2. Cơ hội học tập, nâng cao trình độ sau tốt nghiệp**
        *   Có khả năng tự nghiên cứu và cập nhật công nghệ mới về lĩnh vực công nghệ thông tin để nâng cao trình độ và đáp ứng yêu cầu công việc thực tiễn.
  

## Out-of-scope Query Test
Cell tiep theo dung de test cau hoi ngoai tai lieu (ky vong fallback).

In [8]:
out_of_scope_result = ask("Thoi tiet hom nay o TP.HCM nhu the nao?")

Question: Thoi tiet hom nay o TP.HCM nhu the nao?
Answer: Tôi không tìm thấy thông tin này trong tài liệu.
Sources:
  - CAMNANGTSVSGU2022_ocr.pdf - trang 18
  - CAMNANGTSVSGU2022_ocr.pdf - trang 9
  - BanMOTa_CNTT_2020-2024_ocr.pdf - trang 44
  - CAMNANGTSVSGU2022_ocr.pdf - trang 1

Top retrieved passages:
  1. CAMNANGTSVSGU2022_ocr.pdf - trang 18: . [Ej] TRAM Y TE SO dién thoai: (028) 39 231142 Thu dién tu: yte@sgu.edu.vn Nhiém vu: - Cham séc suc khoe ban dau, so cap cuu, chuyén tuyén ky thuat cho CB, GV, CNV va SV thudc Tru...
  2. CAMNANGTSVSGU2022_ocr.pdf - trang 9: . Hang ngay, phong Dao tao sé thong ké sé ludng nguyén vong, gui Khoa/Nganh xem xét quyét dinh mé thém so ludgng hay khéng. Thdi diém ms thém sé lugng dang ky déi v6i cac nhom hoc ...
  3. BanMOTa_CNTT_2020-2024_ocr.pdf - trang 44: . Bén canh d6, m6n hoc con trang bi cho sinh vién céc k¥ nang van dung tu duy Todn hoc, lap ké hoach va quan Ii thdi gian trong thuc hién chu dé nghién ciru, déng thdi gitip sinh r...
  4. CA